<a href="https://colab.research.google.com/github/seth-coder-cmyk/DH-Project/blob/main/TagbasedArticleComparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Tag-based Article Comparison - The Final JSD Pipeline

For every article pair:

1. Build the union of tags from both language versions.
2. Create two vectors aligned to that union.
3. Fill missing tags with 0.
4. Normalize each vector so the values sum to 1.
5. Pass the two probability vectors to a Jensen-Shannon divergence function.

In [1]:
import json
from scipy.spatial.distance import jensenshannon

#Parse-Weighted_Tags Function
converts weighted_tag list into a union of all tags across both articles.

In [2]:
def parse_weighted_tags(article):
  tags = {}

  for entry in article["weighted_tags"]:
        tag, weight = entry.rsplit("|", 1)
        tags[tag] = float(weight)

  return tags

#Build_Probability_Vectors Function
Creates aligned probability vectors for article_a and article_b

In [3]:
def build_probability_vectors(article_a, article_b):

  tags_a = parse_weighted_tags(article_a)
  tags_b = parse_weighted_tags(article_b)
  #Global tag set (union)
  all_tags = sorted(set(tags_a) | set(tags_b))

  #Extract weights
  vec_a = [tags_a.get(tag, 0.0) for tag in all_tags]
  vec_b = [tags_b.get(tag, 0.0) for tag in all_tags]

  #Normalize
  sum_a = sum(vec_a)
  sum_b = sum(vec_b)

  prob_a = [v / sum_a for v in vec_a]
  prob_b = [v / sum_b for v in vec_b]

  return all_tags, prob_a, prob_b


#Compute_JSD Function
Here, the JSD gets applied and computed.

In [4]:
def compute_jsd(article_a, article_b):
    _, p, q = build_probability_vectors(article_a, article_b)

    #scipy returns sqrt(JSD) by default
    js_distance = jensenshannon(p, q)

    #divergence
    js_divergence = js_distance ** 2

    return js_divergence

#Example

In [5]:
article_als = {
    "page_id": "57327",
    "page_title": "Prähistorische Pfahlbauten um die Alpen",
    "weighted_tags": ["classification.prediction.articlecountry/Switzerland|1000",
                      "classification.prediction.articlecountry/Slovenia|1000",
                      "classification.prediction.articlecountry/Germany|1000",
                      "classification.prediction.articlecountry/Austria|1000",
                      "classification.prediction.articlecountry/France|1000",
                      "classification.prediction.articlecountry/Italy|1000",
                      "classification.prediction.articletopic/Geography.Regions.Europe.Europe*|781",
                      "classification.prediction.articletopic/Geography.Regions.Europe.Western_Europe|988"
                      ]
}

article_de = {
    "page_id": "6307013",
    "page_title": "Prähistorische Pfahlbauten um die Alpen",
"weighted_tags": ["classification.prediction.articlecountry/Switzerland|771",
                  "classification.prediction.articlecountry/Austria|666",
                  "classification.prediction.articlecountry/France|1000",
                  "classification.prediction.articlecountry/Slovenia|666",
                  "classification.prediction.articlecountry/Germany|333",
                  "classification.prediction.articlecountry/Italy|1000",
                  "classification.prediction.articletopic/Geography.Regions.Europe.Europe*|990",
                  "classification.prediction.articletopic/Geography.Geographical|705",
                  "classification.prediction.articletopic/Geography.Regions.Europe.Western_Europe|963"
                  ]
}

#Using the implementation

In [6]:
tags, p, q = build_probability_vectors(article_als, article_de)

print(tags)
print(p)
print(q)

print(compute_jsd(article_als, article_de))

['classification.prediction.articlecountry/Austria', 'classification.prediction.articlecountry/France', 'classification.prediction.articlecountry/Germany', 'classification.prediction.articlecountry/Italy', 'classification.prediction.articlecountry/Slovenia', 'classification.prediction.articlecountry/Switzerland', 'classification.prediction.articletopic/Geography.Geographical', 'classification.prediction.articletopic/Geography.Regions.Europe.Europe*', 'classification.prediction.articletopic/Geography.Regions.Europe.Western_Europe']
[0.1287166945552838, 0.1287166945552838, 0.1287166945552838, 0.1287166945552838, 0.1287166945552838, 0.1287166945552838, 0.0, 0.10052773844767667, 0.12717209422062042]
[0.09388215393290104, 0.14096419509444602, 0.04694107696645052, 0.14096419509444602, 0.09388215393290104, 0.10868339441781788, 0.09937975754158444, 0.13955455314350154, 0.13574851987595152]
0.049438667256242644
